# 04 — Пайплайн: американские акции (USA)

Полный прогон стратегии репликации/арбитража на дневных данных американского рынка.

**Данные:** дневные скорректированные цены закрытия из `data/usa_data.zip`, период 2016-01-01 — 2026-01-01.

**Настройки по умолчанию:**
- Target: `CSCO`
- Universe: первые 25 символов из загруженных цен (исключая target)

**Что делает:**
- Загружает конфиг и цены через `load_run_config("usa")` / `load_market_prices("usa")`
- Запускает `run_unified_pipeline` — PCA-репликация, генерация сигналов, бэктест
- Выводит покрытие (`result.coverage`), состав юниверса и дашборд метрик

**Предупреждение о bias:** юниверс формируется из топ-100 компаний S&P 500 по капитализации в 2016 и 2026 годах одновременно. Использование состава 2026 года при изучении более ранних периодов может завышать качество бэктеста (look-ahead bias).

In [ ]:
from sarb.reporting import display_metrics_table, plot_pipeline_dashboard
from sarb.run import (
    load_market_prices,
    load_run_config,
    metrics_dataframe,
    run_unified_pipeline,
)

run_config = load_run_config("usa")
prices = load_market_prices("usa")
target = "CSCO"
universe = [symbol for symbol in prices.columns if symbol != target][:25]
result = run_unified_pipeline(
    prices,
    target=target,
    universe=universe,
    run_config=run_config,
)

In [ ]:
result.coverage

selection = {
    "target": result.target,
    "universe_size": len(result.universe),
    "universe": ", ".join(result.universe),
}
selection

In [ ]:
plot_pipeline_dashboard(result)
display_metrics_table(metrics_dataframe(result.backtest.metrics))